# Fine Tuning Mistral to improve CMR report generation (DPO)



In [ ]:
# Clone the repo (skip if already cloned)
!git clone https://github.com/JeremieTarantop/Cardiac-Diagnostic-CMR-report-.git repo_cmr
%cd repo_cmr

Cloning into 'repo_cmr'...
remote: Enumerating objects: 15854, done.
remote: Counting objects: 100% (15854/15854), done.
remote: Compressing objects: 100% (15106/15106), done.
remote: Total 15854 (delta 1098), reused 14930 (delta 209), pack-reused 0 (from 0)
Receiving objects: 100% (15854/15854), 34.38 MiB | 19.15 MiB/s, done.
Resolving deltas: 100% (1098/1098), done.
/content/repo_cmr/repo_cmr


In [ ]:
# Install dependencies and enable GPU
import os
os.environ["USE_TRANSFORMERS"] = "1"
os.environ["USE_CUDA"] = "1"

!pip install -q transformers torch pandas numpy scipy

In [ ]:
# Install, the quotes matter
!pip -q install -U "bitsandbytes>=0.46.1" trl datasets accelerate peft transformers

# Verify bitsandbytes is actually importable and the version is correct
import bitsandbytes as bnb, importlib
print("bitsandbytes version:", bnb.__version__)
print("bitsandbytes path:", bnb.__file__)


bitsandbytes version: 0.49.2
bitsandbytes path: /usr/local/lib/python3.12/dist-packages/bitsandbytes/__init__.py


In [ ]:
!pip -q install -U "trl>=0.9.6" "transformers>=4.41.0" "accelerate>=0.31.0" datasets peft bitsandbytes
import trl, transformers, accelerate
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)


trl: 0.29.0
transformers: 5.2.0
accelerate: 1.12.0


#### Transforming an output into a correct judge json file

In [ ]:
import re
import json
from pathlib import Path

def make_judge_jsonl_from_stdout(text: str, out_path: str | Path) -> Path:
    """
    Convert lines like:
      [1/100] 40056908 winner=A
    into JSONL like:
      {"id":"40056908","judgment":{"winner":"A","good_report":"A","bad_report":"B"}}

    Handles duplicated blocks and missing newlines.
    """
    out_path = Path(out_path)

    # Find all occurrences robustly, even if lines are glued together
    pattern = re.compile(r"\[\s*\d+\s*/\s*\d+\s*\]\s*(\d+)\s+winner\s*=\s*([AB])")
    matches = pattern.findall(text)

    # Deduplicate by id, keep first occurrence
    seen = set()
    rows = []
    for rec_id, winner in matches:
        if rec_id in seen:
            continue
        seen.add(rec_id)

        good = winner
        bad = "B" if winner == "A" else "A"

        rows.append({
            "id": rec_id,
            "judgment": {
                "winner": winner,
                "good_report": good,
                "bad_report": bad
            }
        })

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"Wrote {len(rows)} rows to {out_path}")
    return out_path



# ---- Example usage ----
judge_stdout_text = """
[1/100] 40056908 winner=A
[2/100] 40067704 winner=A
[3/100] 40341634 winner=A
[4/100] 40539087 winner=B
[5/100] 40689238 winner=A
[6/100] 40695233 winner=A
[7/100] 40742173 winner=B
[8/100] 40746075 winner=A
[9/100] 40761639 winner=A
[10/100] 40816324 winner=A
[11/100] 40894787 winner=A
[12/100] 40925049 winner=A
[13/100] 41103859 winner=A
[14/100] 41309390 winner=B
[15/100] 41316269 winner=A
[16/100] 41366957 winner=B
[17/100] 41420867 winner=A
[18/100] 41445586 winner=A
[19/100] 41557927 winner=A
[20/100] 41798696 winner=B
[21/100] 41800350 winner=A
[22/100] 41997536 winner=B
[23/100] 42060926 winner=A
[24/100] 42178586 winner=B
[25/100] 42186899 winner=A
[26/100] 42315368 winner=A
[27/100] 42368611 winner=A
[28/100] 42413061 winner=A
[29/100] 42546971 winner=B
[30/100] 42695383 winner=A
[31/100] 42709053 winner=A
[32/100] 42742896 winner=B
[33/100] 42790963 winner=A
[34/100] 42855628 winner=A
[35/100] 42947358 winner=A
[36/100] 43013697 winner=A
[37/100] 43143228 winner=A
[38/100] 43468501 winner=A
[39/100] 43492795 winner=B
[40/100] 43522917 winner=B
[41/100] 43576724 winner=A
[42/100] 43647002 winner=B
[43/100] 43681375 winner=B
[44/100] 43742645 winner=A
[45/100] 43744967 winner=B
[46/100] 43749278 winner=A
[47/100] 44069449 winner=B
[48/100] 44095784 winner=B
[49/100] 44458630 winner=A
[50/100] 44669583 winner=A
[51/100] 44837313 winner=B
[52/100] 44881309 winner=A
[53/100] 44986768 winner=A
[54/100] 45090959 winner=A
[55/100] 45105401 winner=B
[56/100] 45386375 winner=B
[57/100] 45594411 winner=A
[58/100] 45808859 winner=B
[59/100] 45832921 winner=A
[60/100] 45842645 winner=A
[61/100] 45896667 winner=A
[62/100] 45986118 winner=A
[63/100] 46034277 winner=A
[64/100] 46049322 winner=A
[65/100] 46293961 winner=B
[66/100] 46440135 winner=A
[67/100] 46543372 winner=B
[68/100] 46566322 winner=B
[69/100] 46642833 winner=A
[70/100] 46743816 winner=A
[71/100] 46857734 winner=B
[72/100] 47004256 winner=B
[73/100] 47218930 winner=A
[74/100] 47226536 winner=A
[75/100] 47286409 winner=A
[76/100] 47499371 winner=B
[77/100] 47544594 winner=A
[78/100] 47843381 winner=A
[79/100] 47863823 winner=A
[80/100] 48132909 winner=A
[81/100] 48142435 winner=A
[82/100] 48150809 winner=A
[83/100] 48288896 winner=B
[84/100] 48339811 winner=A
[85/100] 48376834 winner=A
[86/100] 48446569 winner=A
[87/100] 48531703 winner=B
[88/100] 48575806 winner=A
[89/100] 48582094 winner=A
[90/100] 48778222 winner=A
[91/100] 49036311 winner=A
[92/100] 49144190 winner=A
[93/100] 49202548 winner=B
[94/100] 49245181 winner=A
[95/100] 49368993 winner=A
[96/100] 49466912 winner=A
[97/100] 49500365 winner=A
[98/100] 49560547 winner=B
[99/100] 49758073 winner=A
[100/100] 49961002 winner=A
"""

make_judge_jsonl_from_stdout(judge_stdout_text, Path("data") / "judge_results_100.jsonl")

Wrote 100 rows to data/judge_results_100.jsonl


PosixPath('data/judge_results_100.jsonl')

## Creating a correct dataset for DPO framework

In [ ]:
from pathlib import Path
import json
from datasets import Dataset

ROOT = Path(".")
MEETI_ROOT = ROOT / "data" / "20260209_MEETI TXT"
PROMPT_PATH = ROOT / "prompt_v1.txt"
GEN_DIR = MEETI_ROOT / "generated_reports"

JUDGE_JSONL = ROOT / "data" / "judge_results_100.jsonl"

assert MEETI_ROOT.exists(), f"Missing: {MEETI_ROOT}"
assert GEN_DIR.exists(), f"Missing: {GEN_DIR}"
assert JUDGE_JSONL.exists(), f"Missing: {JUDGE_JSONL}"
assert PROMPT_PATH.exists(), f"Missing: {PROMPT_PATH}"


In [ ]:
import re

def extract_response_only(text: str) -> str:
    """
    Extracts text after the SECOND occurrence of 'Response:' or '**Response:**'.
    This avoids keeping prompt scaffolding when the prompt itself contains 'Response:'.
    """

    if not text:
        return ""

    # Find all occurrences of markdown or plain Response:
    pattern = r"\*\*Response:\*\*|\bResponse:"
    matches = list(re.finditer(pattern, text))

    if len(matches) >= 2:
        # Cut after the second occurrence
        second = matches[1]
        return text[second.end():].strip()

    elif len(matches) == 1:
        # Fallback: only one found
        return text[matches[0].end():].strip()

    # If none found, return original
    return text.strip()

In [ ]:
def safe_read_text(path: Path) -> str | None:
    try:
        return path.read_text(encoding="utf-8").strip()
    except Exception:
        return None


def split_prompt_and_response(text: str):

    """
    Returns:
        prompt_part (everything before the relevant Response marker)
        response_part (everything after the correct Response marker)

    Logic:
    - If 2+ occurrences of 'Response:' exist → split at the SECOND
    - If 1 occurrence → split at that one
    - If none → return (None, None)
    """

    if not text:
        return None, None

    pattern = r"\*\*Response:\*\*|\bResponse:"
    matches = list(re.finditer(pattern, text))

    if not matches:
        return None, None

    # If there are 2 or more, use the second
    if len(matches) >= 2:
        split_point = matches[1].start()
        response_start = matches[1].end()
    else:
        split_point = matches[0].start()
        response_start = matches[0].end()

    prompt_part = text[:split_point].strip()
    response_part = text[response_start:].strip()

    return prompt_part, response_part


In [ ]:
from pathlib import Path

def find_llm_interpretation_path(meeti_root: Path, rec_id: str) -> Path | None:
    """
    Find the nested MEETI llm interpretation file for a given record id.

    Example expected filename:
      {rec_id}_llm_interpretation.txt

    Searches recursively under meeti_root (covers p*/p*/s*/ structure).
    """
    pattern = f"**/{rec_id}_llm_interpretation.txt"
    matches = list(meeti_root.glob(pattern))
    if not matches:
        return None
    # If duplicates exist, pick the first deterministic one
    return sorted(matches)[0]


def load_llm_interpretation_text(meeti_root: Path, rec_id: str) -> str | None:
    """
    Find and read the llm interpretation text for rec_id.
    Returns stripped text, or None if not found/unreadable/empty.
    """
    path = find_llm_interpretation_path(meeti_root, rec_id)
    if path is None:
        return None
    return safe_read_text(path)

In [ ]:
# Importing the good halluncinated reports

from pathlib import Path

# Directory with your 100 “better hallucinations”
GOOD_GEN_DIR = MEETI_ROOT / "Generated_with_5w_ECGreport"

def iter_good_pairs(gen_dir: Path):

    if not gen_dir.exists():
        print(f"[WARN] GOOD_GEN_DIR not found: {gen_dir}")
        return

    # Most robust: accept any .txt and extract id as first token before "_"
    for gf in sorted(gen_dir.glob("*.txt")):
        rec_id = gf.stem.split("_")[0]
        raw = safe_read_text(gf)
        if not raw:
            continue
        yield {"id": rec_id, "gen_text_raw": raw}


In [ ]:
examples = []

import os
from pathlib import Path


with open(JUDGE_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        rec_id = row["id"]
        winner = row.get("judgment", {}).get("winner")  # "A" or "B"
        if winner not in ("A", "B"):
            continue

        # Load generated reports and hallucinated reports
        gen_path = GEN_DIR / f"{rec_id}_hallucinated.txt"
        if not gen_path.exists():
            print("The gen_path is not found for the hallucinated files")
        gen_text_raw = safe_read_text(gen_path)

        llm_text = load_llm_interpretation_text(MEETI_ROOT, rec_id)
        if not llm_text:
            continue

        prompt, gen_report = split_prompt_and_response(gen_text_raw)
        if not prompt or not gen_report:
            continue

        # A = MEETI llm interpretation, B = generated hallucination
        if winner == "A":
            chosen, rejected = llm_text, gen_report
        else:
            chosen, rejected = gen_report, llm_text

        examples.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})


# train_ds = Dataset.from_list(examples)
print("Total examples:", len(examples))
print("Prompt preview:", examples[0]["prompt"][:100])
print("Chosen preview:", examples[0]["chosen"][:100])
print("Rejected preview:", examples[0]["rejected"][:100])

Total examples: 100
Prompt preview: Follow the user instructions exactly.

A.1 Diagnosis Guider Prompt 
# Your task:  Interpret the prov
Chosen preview: Upon examining the 12-lead ECG, several key characteristics and abnormalities are noted. The rhythm 
Rejected preview: Upon thorough analysis of the provided ECG data, several key features and abnormalities are apparent


In [ ]:
# Add the extra 200 pairs (LLM interpretation always "chosen")
added_good = 0
skipped_good = 0
skipped_llm_text = 0

for item in iter_good_pairs(GOOD_GEN_DIR):

    rec_id = item["id"]
    report_path = MEETI_ROOT / f"{rec_id}_report.txt"
    gen_text_raw = item["gen_text_raw"]

    llm_text = load_llm_interpretation_text(MEETI_ROOT, rec_id)

    if not llm_text:
        skipped_good += 1
        skipped_llm_text += 1
        continue


    prompt, gen_report = split_prompt_and_response(gen_text_raw)
    if not prompt or not gen_report:
        skipped_good += 1
        continue

    examples.append({"prompt": prompt, "chosen": llm_text, "rejected": gen_report})
    added_good += 1

print(f"Added good pairs: {added_good}, skipped: {skipped_good}")

train_ds = Dataset.from_list(examples)
print("Total examples:", len(examples))
print("Compteur:", compteur)

print("Example keys:", examples[0].keys())
print("Prompt preview:", examples[0]["prompt"][:300])
print("Chosen preview:", examples[0]["chosen"][:300])
print("Rejected preview:", examples[0]["rejected"][:300])
print(train_ds[0].keys())

Added good pairs: 206, skipped: 0
Total examples: 306
Compteur: 206
Example keys: dict_keys(['prompt', 'chosen', 'rejected'])
Prompt preview: Follow the user instructions exactly.

A.1 Diagnosis Guider Prompt 
# Your task:  Interpret the provided ECG data, identify key features and abnormalities in each lead, and generate a clinical diagnosis that is supported by the observed evidence. 

## Key objectives:
1.  Simulate a Realistic Diagnos
Chosen preview: Upon examining the 12-lead ECG, several key characteristics and abnormalities are noted. The rhythm appears irregular due to the absence of distinct P waves, suggesting atrial fibrillation. The QRS complexes show an increased duration, particularly evident in Lead V1 with a rsR' pattern indicating a
Rejected preview: Upon thorough analysis of the provided ECG data, several key features and abnormalities are apparent in various leads.

Initially, in Lead I, the ST segment exhibits a subtle elevation, and the T wave morphology is somewha

#### Set up TRL DPOTrainer with a frozen reference model

Downloading MedGemma as a potential reference model

In [ ]:
# from huggingface_hub import hf_hub_download

# token = "hf_XXX"
# hf_hub_download(
#     repo_id="google/medgemma-4b-it",
#     filename="config.json",
#     token=token
# )
# print("Access confirmed")


In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer


MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# cfg = AutoConfig.from_pretrained(MODEL_ID)
# cfg.tie_word_embeddings = False

# 2) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3) Load policy model on GPU (trainable via LoRA)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
)

# 4) Load reference model on CPU (frozen anchor)
model_ref = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
)

model_ref.eval()
for p in model_ref.parameters():
    p.requires_grad_(False)

# # 5) LoRA on policy model only
# peft_cfg = LoraConfig(
#     r=16,
#     lora_alpha=16,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
# )

# 5.2) LoRA on MLP too. According to ChatGPT is is better but more computationaly intensive
peft_cfg = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

# # 6) DPO config
# dpo_args = DPOConfig(
#     output_dir="dpo_cmr_sanity",
#     beta=0.1,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=5,
#     num_train_epochs=2,
#     learning_rate=1e-5,
#     logging_steps=10,
#     save_steps=50,
#     fp16=True,
#     gradient_checkpointing=True,
# )

# If you want to train on the entire dataset

# # 7) Trainer
# trainer_kwargs = dict(
#     model=model,
#     ref_model=model_ref,
#     args=dpo_args,
#     train_dataset=train_ds,
#     peft_config=peft_cfg,
# )

# try:
#     trainer = DPOTrainer(**trainer_kwargs, processing_class=tokenizer)
# except TypeError:
#     trainer = DPOTrainer(**trainer_kwargs, tokenizer=tokenizer)

# print("Ready. Next: trainer.train()")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

#### If you want to train on 80% of the dataset and test on the remaining 20% later

In [ ]:
# Shuffle and split 80 / 20
train_ds = train_ds.shuffle(seed=42)

# Choose split fraction
train_frac = 0.8  # 80% train, 20% eval

n = len(train_ds)
n_train = int(n * train_frac)
n_eval = n - n_train

train_split = train_ds.select(range(n_train))
eval_split  = train_ds.select(range(n_train, n))

print("Total:", n)
print("Train size:", len(train_split))
print("Eval size:", len(eval_split))

# 6) DPO config
dpo_args = DPOConfig(
    output_dir="dpo_cmr_sanity",
    beta=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=5,
    num_train_epochs=3,
    learning_rate=1e-5,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=25,
)

# 7) Trainer
trainer_kwargs = dict(
    model=model,
    ref_model=model_ref,
    args=dpo_args,
    train_dataset=train_split,
    eval_dataset=eval_split,
    peft_config=peft_cfg,
)

try:
    trainer = DPOTrainer(**trainer_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = DPOTrainer(**trainer_kwargs, tokenizer=tokenizer)

print("Ready. Next: trainer.train()")

Total: 306
Train size: 244
Eval size: 62


Adding EOS to train dataset:   0%|          | 0/244 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/244 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]

Ready. Next: trainer.train()


## Training Part

In [ ]:
# Optional: print trainable params (LoRA)
trainer.model = trainer._wrap_model(trainer.model, training=True)

def count_params(m):
    total = 0
    trainable = 0
    for p in m.parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
    return total, trainable

total_params, trainable_params = count_params(trainer.model)
ratio = trainable_params / total_params
print(f"\nTotal parameters:      {total_params:,}")
print(f"Trainable parameters:  {trainable_params:,}")
print(f"Trainable ratio:       {ratio:.6%}")


Total parameters:      7,289,966,592
Trainable parameters:  41,943,040
Trainable ratio:       0.575353%


In [ ]:
# Train DPO
import gc
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_out = trainer.train()
print(train_out)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
25,0.413344,0.545754
50,0.568182,0.531949
75,0.192330,0.330854
100,0.143639,0.356022
125,0.067608,0.388022


TrainOutput(global_step=147, training_loss=0.23856960166068303, metrics={'train_runtime': 753.5081, 'train_samples_per_second': 0.971, 'train_steps_per_second': 0.195, 'total_flos': 9.216687543518822e+16, 'train_loss': 0.23856960166068303})


## Evaluation part

In [ ]:
# Evaluate on held-out split
metrics = trainer.evaluate()
print("\n===== RAW EVAL METRICS DICT =====")
print(metrics)

# Print the key DPO signals (keys differ slightly by TRL version, so probe safely)
def pick(metrics_dict, candidates):
    for k in candidates:
        if k in metrics_dict:
            return k, metrics_dict[k]
    return None, None

acc_k, acc_v = pick(metrics, ["eval_rewards/accuracies", "rewards/accuracies", "eval_reward_accuracy", "reward_accuracy"])
mar_k, mar_v = pick(metrics, ["eval_rewards/margins", "rewards/margins", "eval_reward_margin", "reward_margin"])
loss_k, loss_v = pick(metrics, ["eval_loss", "loss"])

print("\n===== DPO HELD-OUT SUMMARY =====")
print(f"{acc_k}: {acc_v}")
print(f"{mar_k}: {mar_v}")
print(f"{loss_k}: {loss_v}")


===== RAW EVAL METRICS DICT =====
{'eval_loss': 0.37290504574775696, 'eval_runtime': 21.9903, 'eval_samples_per_second': 2.819, 'eval_steps_per_second': 0.364}

===== DPO HELD-OUT SUMMARY =====
None: None
None: None
eval_loss: 0.37290504574775696


In [ ]:
from transformers import AutoModelForCausalLM

# Reload clean base policy (no LoRA)
base_policy = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_policy.eval()

# Baseline trainer (no PEFT)
baseline_trainer_kwargs = dict(
    model=base_policy,
    ref_model=model_ref,
    args=dpo_args,
    train_dataset=train_split,
    eval_dataset=eval_split,
)

try:
    baseline_trainer = DPOTrainer(**baseline_trainer_kwargs, processing_class=tokenizer)
except TypeError:
    baseline_trainer = DPOTrainer(**baseline_trainer_kwargs, tokenizer=tokenizer)

# ---- Evaluate baseline and tuned ----
base_metrics = baseline_trainer.evaluate()
tuned_metrics = trainer.evaluate()

# Robust key picker (TRL version safe)
def pick(metrics_dict, candidates):
    for k in candidates:
        if k in metrics_dict:
            return metrics_dict[k]
    return None

# Extract metrics safely
base_acc = pick(base_metrics, ["eval_rewards/accuracies", "rewards/accuracies"])
base_margin = pick(base_metrics, ["eval_rewards/margins", "rewards/margins"])
base_loss = pick(base_metrics, ["eval_loss", "loss"])

tuned_acc = pick(tuned_metrics, ["eval_rewards/accuracies", "rewards/accuracies"])
tuned_margin = pick(tuned_metrics, ["eval_rewards/margins", "rewards/margins"])
tuned_loss = pick(tuned_metrics, ["eval_loss", "loss"])

print("\n===== HELD-OUT COMPARISON =====")

print("\nBASELINE (pre fine-tune)")
print(f"Preference accuracy: {base_acc * 100:.1f}%")
print(f"Reward margin:       {base_margin:.2f}")
print(f"Eval loss:           {base_loss:.3f}")

print("\nTUNED (post fine-tune)")
print(f"Preference accuracy: {tuned_acc * 100:.1f}%")
print(f"Reward margin:       {tuned_margin:.2f}")
print(f"Eval loss:           {tuned_loss:.3f}")

print("\nDELTA (Tuned - Base)")
print(f"Δ accuracy: {(tuned_acc - base_acc) * 100:.1f}%")
print(f"Δ margin:   {(tuned_margin - base_margin):.2f}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/244 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/244 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]


===== HELD-OUT COMPARISON =====

BASELINE (pre fine-tune)


TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'


## Visualization of "Before" and "After" finetuning

In [ ]:
import torch

@torch.inference_mode()
def generate_from_prompt(model, tokenizer, prompt: str, max_new_tokens: int = 1000, temperature: float = 0.7, top_p: float = 0.95):
    """
    Generates a completion from a plain prompt using the model's chat template.
    Uses the same generation style you used before.
    """
    model.eval()
    messages = [
        {"role": "user", "content": prompt},
    ]

    chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded[len(chat):].strip() if decoded.startswith(chat) else decoded.strip()



In [ ]:
# Comparing the first report

def compare_first_pair(train_ds, tokenizer, base_model, trainer, max_new_tokens: int = 1000, temperature: float = 0.7, top_p: float = 0.95):
    """
    Uses the FIRST example in train_ds.

    Prints:
      - report B (train_ds[0]["rejected"])
      - base_model generation (non fine tuned)
      - fine tuned generation (trainer.model, LoRA after DPO)
    """

    ex = train_ds[0]
    prompt_v1 = ex["prompt"]
    report_b = ex["rejected"]

    base_gen = generate_from_prompt(
        base_model, tokenizer, prompt_v1,
        max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p
    )

    tuned_gen = generate_from_prompt(
        trainer.model, tokenizer, prompt_v1,
        max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p
    )


    print("\n" + "=" * 120)
    print("REPORT B (train_ds[0]['rejected'])")
    print("=" * 120)
    print(report_b)

    print("\n" + "=" * 120)
    print("BASE MODEL GENERATION (non fine tuned Mistral)")
    print("=" * 120)
    print(base_gen)

    print("\n" + "=" * 120)
    print("FINE TUNED GENERATION (DPO + LoRA, trainer.model)")
    print("=" * 120)
    print(tuned_gen)

    return {
        "report_b": report_b,
        "base_gen": base_gen,
        "tuned_gen": tuned_gen,
    }

In [ ]:
# result = compare_first_pair(
#     train_ds=train_ds,
#     tokenizer=tokenizer,
#     base_model=model,
#     trainer=trainer,
#     max_new_tokens=2000,
#     temperature=0.7,
#     top_p=0.95,
# )

## Generating 10 reports with the fine tuned version

In [ ]:
import json
import zipfile
from pathlib import Path
import torch

def export_finetuned_hallucinations_zip(
    train_ds,
    trainer,
    tokenizer,
    judge_jsonl_path,
    n: int = 100,
    out_zip_path: str = "hallucinated_fine_tuned_firstN.zip",
    max_new_tokens: int = 1000,
    temperature: float = 0.7,
    top_p: float = 0.95,
):
    """
    Generates N new reports using:
      - prompt = train_ds[i]["prompt"]
      - model = trainer.model (fine-tuned)

    Uses the first N IDs directly from the JUDGE_JSONL file.
    Saves only the new reports into a zip file named:
      {rec_id}_hallucinated_fine_tuned.txt
    """

    n = min(n, len(train_ds))
    out_zip_path = Path(out_zip_path)

    # Read first N IDs from JSONL
    rec_ids = []
    with open(judge_jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rec_ids.append(str(row["id"]))
            if len(rec_ids) >= n:
                break

    if len(rec_ids) < n:
        raise ValueError("Not enough IDs found in judge JSON file.")

    @torch.inference_mode()
    def generate_from_prompt(model, tokenizer, prompt):
        model.eval()
        messages = [{"role": "user", "content": prompt}]
        chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(chat, return_tensors="pt").to(model.device)

        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        decoded = tokenizer.decode(out[0], skip_special_tokens=True)
        return decoded[len(chat):].strip() if decoded.startswith(chat) else decoded.strip()

    with zipfile.ZipFile(out_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for i in range(n):
            rec_id = rec_ids[i]
            prompt = train_ds[i]["prompt"]

            tuned_text = generate_from_prompt(trainer.model, tokenizer, prompt)

            fname = f"{rec_id}_hallucinated_fine_tuned.txt"
            zf.writestr(fname, tuned_text.strip() + "\n")

            print(f"[{i+1}/{n}] wrote {fname}")

    print(f"\nSaved zip to: {out_zip_path.resolve()}")
    return str(out_zip_path)

In [ ]:
zip_path = export_finetuned_hallucinations_zip(
    train_ds=train_ds,
    trainer=trainer,
    tokenizer=tokenizer,
    judge_jsonl_path=JUDGE_JSONL,
    n=100,
    out_zip_path="hallucinated_fine_tuned_first100.zip",
    max_new_tokens=1000,
)

print("ZIP ready at:", zip_path)

[1/100] wrote 40056908_hallucinated_fine_tuned.txt
[2/100] wrote 40067704_hallucinated_fine_tuned.txt
[3/100] wrote 40341634_hallucinated_fine_tuned.txt
[4/100] wrote 40539087_hallucinated_fine_tuned.txt
[5/100] wrote 40689238_hallucinated_fine_tuned.txt
[6/100] wrote 40695233_hallucinated_fine_tuned.txt
[7/100] wrote 40742173_hallucinated_fine_tuned.txt
[8/100] wrote 40746075_hallucinated_fine_tuned.txt
[9/100] wrote 40761639_hallucinated_fine_tuned.txt
[10/100] wrote 40816324_hallucinated_fine_tuned.txt
[11/100] wrote 40894787_hallucinated_fine_tuned.txt
[12/100] wrote 40925049_hallucinated_fine_tuned.txt
[13/100] wrote 41103859_hallucinated_fine_tuned.txt
[14/100] wrote 41309390_hallucinated_fine_tuned.txt
[15/100] wrote 41316269_hallucinated_fine_tuned.txt
[16/100] wrote 41366957_hallucinated_fine_tuned.txt
[17/100] wrote 41420867_hallucinated_fine_tuned.txt
[18/100] wrote 41445586_hallucinated_fine_tuned.txt
[19/100] wrote 41557927_hallucinated_fine_tuned.txt
[20/100] wrote 417986

In [ ]:
from google.colab import files
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Saving the finetuned model

In [ ]:
# After trainer.train()

save_dir = "mistral_cmr_dpo_lora"

trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"LoRA adapters saved to: {save_dir}")
